In [36]:
# Session Setup
# -------------

# Boilerplate Modules
import os
import sys
import json

from dotenv import load_dotenv

import time

from tqdm import tqdm

# Modelus to Build Corpus from Corpora Scrapped from the Web
import wikipediaapi
import arxiv
from sec_edgar_downloader import Downloader
from firecrawl import FirecrawlApp


In [37]:
# Load variables from .env into the environment
load_dotenv()

True

In [39]:
firecrawl_api_key = os.getenv("FIRECRAWL_KEY")

Defining topics

In [25]:
pages = [
    "Attention_Is_All_You_Need", 
    "BERT_(language_model)", 
    "GPT-4", 
    "Retrieval-augmented_generation", 
    "LoRA_(machine learning)",
    "Snowflake_Inc.",
    "Apache_Airflow",
    "Apache_Spark",
    "Apache_Beam",
    "Apache_Kafka",
    "ChromaDB",
    "Vector_database",
    "Dask_(software)",
    "Embedding_(machine_learning)",
    "Money_laundering",
    "Data_analysis_for_fraud_detection",
    "Anti-Money_Laundering_Improvement_Act",
    "JPMorgan_Chase",
    "Capital_One",
    ]

In [42]:
# List of target documentation roots
doc_urls = [
    "https://airflow.apache.org/docs/",
    # "https://spark.apache.org/docs/latest/",
    # "https://docs.snowflake.com/en/",
    # "https://beam.apache.org/documentation/",
    # "https://kafka.apache.org/documentation/"
]

Wikipedia

In [7]:
# Setting up an user-agent
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rics-rag-project/1.0 (rzambrano@gmail.com)'
)

In [26]:

wikipedia_docs = []

for title in tqdm(pages, desc="Pulling from Wikipedia...", colour='green'):
    page = wiki.page(title)
    if page.exists():
        wikipedia_docs.append({"title": title, "text": page.text})
        print(f"Ok... {title} ({len(page.text)} chars)")
    else:
        print(f"Oh No!... {title} — not found")
    print("-" * 50)

Pulling from Wikipedia...:  11%|█         | 2/19 [00:00<00:03,  5.65it/s]

Ok... Attention_Is_All_You_Need (19247 chars)
--------------------------------------------------
Ok... BERT_(language_model) (17402 chars)
--------------------------------------------------


Pulling from Wikipedia...:  21%|██        | 4/19 [00:00<00:02,  5.63it/s]

Ok... GPT-4 (14513 chars)
--------------------------------------------------
Ok... Retrieval-augmented_generation (11462 chars)
--------------------------------------------------


Pulling from Wikipedia...:  32%|███▏      | 6/19 [00:00<00:02,  6.35it/s]

...

Ok... Money_laundering (33331 chars)
--------------------------------------------------
Ok... Data_analysis_for_fraud_detection (10261 chars)
--------------------------------------------------


Pulling from Wikipedia...:  95%|█████████▍| 18/19 [00:03<00:00,  6.16it/s]

Ok... Anti-Money_Laundering_Improvement_Act (4989 chars)
--------------------------------------------------
Ok... JPMorgan_Chase (67477 chars)
--------------------------------------------------


Pulling from Wikipedia...: 100%|██████████| 19/19 [00:03<00:00,  5.86it/s]

Ok... Capital_One (19368 chars)
--------------------------------------------------


In [28]:
wikipedia_docs[-1]

{'title': 'Capital_One',
 'text': 'Capital One Financial Corporation is an American bank holding company headquartered in Tysons, Virginia, with operations in the United States, Canada, and the United Kingdom. One of the largest banks in the United States, it is the largest issuer of credit cards in the United States, and is one of the largest car finance companies in the U.S. It owns the Discover Card, Diners Club, and Pulse payment networks. The company\'s three business segments are credit cards, consumer banking, and commercial banking. It has approximately 750 bank branches, including over 60 café style locations, and 7,000 ATMs in the United States. The company\'s corporate headquarters are in the Capital One Tower and its European headquarters are in Trent House, Nottingham.\nThe company is ranked 82nd on the Fortune 500.\nThe company helped pioneer the mass marketing of credit cards in the 1990s.\n\nHistory\n1994–2004\nRichard Fairbank and Nigel Morris developed the idea of usi

Arxiv

In [29]:
# Setting up an user-agent
arxiv_client = arxiv.Client()

In [30]:
max_arxiv_results = 2
arxiv_docs = []

for title in tqdm(pages, desc="Pulling from Arxiv...", colour='green'):
    query = title.replace("_", " ").lower()
    arxiv_result = arxiv.Search(query=query, max_results=max_arxiv_results)
    for result in arxiv_client.results(arxiv_result):
        if result:
            arxiv_docs.append({"title": result.title, "text": result.summary}) # Using only the abstract
            print(f"Ok... {result.title} ({len(result.summary)} chars)")
        else:
            print(f"Oh No!... something went wrong :'(")
        print("-" * 50)


Pulling from Arxiv...:   5%|▌         | 1/19 [00:00<00:12,  1.47it/s]

Ok... Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet (1121 chars)
--------------------------------------------------
Ok... "All You Need" is Not All You Need for a Paper Title: On the Origins of a Scientific Meme (848 chars)
--------------------------------------------------


Pulling from Arxiv...:  11%|█         | 2/19 [00:03<00:35,  2.07s/it]

Ok... The Road to Know-Where: An Object-and-Room Informed Sequential BERT for Indoor Vision-Language Navigation (1379 chars)
--------------------------------------------------
Ok... TinyBERT: Distilling BERT for Natural Language Understanding (1392 chars)
--------------------------------------------------


Pulling from Arxiv...:  16%|█▌        | 3/19 [00:07<00:44,  2.79s/it]

...

Ok... Anti-Money Laundering Systems Using Deep Learning (1786 chars)
--------------------------------------------------
Ok... Intelligent Anti-Money Laundering Solution Based upon Novel Community Detection in Massive Transaction Networks on Spark (972 chars)
--------------------------------------------------


Pulling from Arxiv...:  95%|█████████▍| 18/19 [01:05<00:03,  3.91s/it]

Ok... Cross comparison and modelling of Goldman Sachs, Morgan Stanley, JPMorgan Chase, Bank of America, and Franklin Resources (565 chars)
--------------------------------------------------
Ok... JEL: Applying End-to-End Neural Entity Linking in JPMorgan Chase (1450 chars)
--------------------------------------------------


Pulling from Arxiv...: 100%|██████████| 19/19 [01:09<00:00,  3.66s/it]

Ok... The Theory of Crowd Capital (795 chars)
--------------------------------------------------
Ok... A Micro-foundation of Social Capital in Evolving Social Networks (1285 chars)
--------------------------------------------------


In [31]:
arxiv_docs[-1]

{'title': 'A Micro-foundation of Social Capital in Evolving Social Networks',
 'text': 'A social network confers benefits and advantages on individuals (and on groups), the literature refers to these advantages as social capital. This paper presents a micro-founded mathematical model of the evolution of a social network and of the social capital of individuals within the network. The evolution of the network is influenced by the extent to which individuals are homophilic, structurally opportunistic, socially gregarious and by the distribution of types in the society. In the analysis, we identify different kinds of social capital: bonding capital, popularity capital, and bridging capital. Bonding capital is created by forming a circle of connections, homophily increases bonding capital because it makes this circle of connections more homogeneous. Popularity capital leads to preferential attachment: individuals who become popular tend to become more popular because others are more likely

SEC

In [20]:
# Setting up an user-agent
dl = Downloader("MyDataScienceProject", "rzambrano@gmail.com")

In [21]:
dl.get("10-K", "JPM", limit=2)

2

In [ ]:
dl.get("10-K", "COF", limit=2)

2

Firecrawl

In [40]:
# Initialize the app
app = FirecrawlApp(api_key=firecrawl_api_key)

In [ ]:
for url in doc_urls:
        print(f"Starting crawl for: {url}")

        response  = app.crawl(
            url, 
            "limit": 100,
            scrape_options={
                'formats': [
                'markdown',
                { 'type': 'json', 'schema': { 'type': 'object', 'properties': { 'title': { 'type': 'string' } } } }
                ],
                'proxy': 'auto',
                'max_age': 600000,
                'only_main_content': True
                }
        )